# 數位控制系統第七章：穩定度分析技巧（教學版 Notebook）

本 Notebook 是 `chp7.md` 教材的教學版，額外補充：

- 每個第一次出現的函數（`d2c`, `rlocus`, `nyquist`, `bode`, `margin`, `freqresp`, `polyder`, ...）的逐步解說
- **三個實測發現的 Octave 陷阱**（`bode` 相位、`margin` 失效、`exist` 判斷不了 LTI 方法）
- 公式 → 程式碼的逐項對照（每段程式都標註對應的教材式號與範例編號）
- 七個可直接執行的實驗，親手驗證本章每一條公式

建議搭配 `chp7.md`（完整理論、推導與符號定義）與 `chp7.m`（精簡可執行版）一起閱讀。

> **本章要回答的問題**：第 6 章告訴我們「極點在哪 = 什麼行為」。本章要在**不解方程式**的前提下判斷**極點到底在不在單位圓內**，以及**增益 $K$ 可以調到多大**。

---


## 🔧 環境設定

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。本章大量使用 `rlocus`、`nyquist`、`bode`、`margin`、`d2c`。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

printf('環境就緒\n');

## 📖 全章符號總表

| 符號 | 意義 |
|---|---|
| $\overline{GH}(z)$ | 開迴路轉移函數（先在 $s$ 域相乘再取 z 轉換） |
| $K$ | 可調的迴路增益 |
| $w$ | **雙線性轉換變數**（$w$ 平面） |
| $\omega_w$ | **$w$ 平面頻率** |
| $\omega$ | 真實頻率（$s$ 平面） |
| $\omega_s$ | 取樣角頻率 $=2\pi/T$ |
| $Q(z)$ | 特徵多項式 |
| $N,P,Z$ | Nyquist 準則：包圍次數、開迴路外部極點數、閉迴路外部極點數 |
| $\eta_m$ | 相位裕度 |

## 📖 三個平面的對照（本章的核心地圖）

| 平面 | 穩定區域 | 能用什麼工具 |
|---|---|---|
| $s$ | 左半平面 | 連續系統的所有工具 |
| $z$ | **單位圓內** | Jury、根軌跡、Nyquist |
| $w$ | **左半平面** | **Routh–Hurwitz、Bode**（因為穩定邊界又變回虛軸） |

## 📖 本章會用到的 Octave 語法小抄

| 語法 | 意義 | 注意事項 |
|---|---|---|
| `d2c(Gz,'tustin')` | 雙線性轉換 $z\to w$ | **`'tustin'` 就是雙線性轉換** |
| `rlocus(Gz)` | 根軌跡 | |
| `nyquist(Gz)` | Nyquist 圖 | |
| `bode(Gw)` | Bode 圖 | **畫圖沒問題；取「相位數值」有陷阱** |
| `[gm,pm,wg,wp]=margin(sys)` | 增益／相位裕度 | **`gm` 是倍率不是 dB** |
| **`freqresp(sys,w)`** | 頻率響應 | **回傳複數，取相位一律用這個** |
| `roots(p)` | 多項式的根 | |
| `polyder(p)` | 多項式微分 | 求根軌跡分離點 |
| `conv(a,b)` | 多項式相乘 | |
| `unwrap(x)` | 展開相位（去掉 $\pm2\pi$ 跳變） | 手動算裕度時必用 |

> **⚠️ 本章三個實測發現的 Octave 陷阱**（實驗 7 附錄會逐一示範）
>
> 1. **`bode` 對「含 $z=1$ 極點」的離散系統，回傳的相位是錯的**（差 $90^\circ\sim180^\circ$）。**取相位一律用 `freqresp`。**
> 2. **`margin` 有時找不到增益交越頻率**，回傳 `pm=180`、`wp=NaN`。遇到就自己掃頻算。
> 3. **`exist('d2c')` 回傳 0，但 `d2c` 可以正常呼叫**——它是 LTI 類別的**方法**而非一般函數。

---


## 二、穩定度與特徵方程式 (7.2)

### 判準

$$C(z)=\frac{k_1z}{z-p_1}+\cdots+\frac{k_nz}{z-p_n}+C_R(z),\qquad
\mathcal{Z}^{-1}\left[\frac{k_iz}{z-p_i}\right]=k_i(p_i)^k$$

$$\boxed{\text{系統穩定}\iff\text{特徵方程式 }1+\overline{GH}(z)=0\text{ 的所有根都在單位圓「內」}}$$

| $| p_i|$ | 該項行為 | 系統 |
|---|---|---|
| $<1$ | $p_i^k\to0$ | **穩定** |
| $=1$ | 大小不變 | **臨界穩定**（marginally stable） |
| $>1$ | 發散 | **不穩定** |

### 寫不出轉移函數時怎麼辦

第 5 章有些系統寫不出轉移函數。教材的做法：

> **令 $R(s)=0$（穩定度與輸入無關），在「某個取樣器的前面」斷開系統，求該處的轉移函數 $G_{op}(z)$。**
>
> **為什麼一定在取樣器處斷開？** 因為只要輸入在進入連續部分前先被取樣，就一定寫得出轉移函數。

$$\boxed{1-G_{op}(z)=0\quad\text{就是特徵方程式}}$$

---


## 三、雙線性轉換 (7.3)

### 為什麼需要它

> **Routh–Hurwitz 與 Bode 技巧都建立在「$s$ 平面的穩定邊界是虛軸」上。$z$ 平面的穩定邊界是單位圓，所以不能直接用。**

**解法**：找一個變換把單位圓變成虛軸。

$$\boxed{z=\frac{1+(T/2)w}{1-(T/2)w}\quad\Longleftrightarrow\quad w=\frac{2}{T}\cdot\frac{z-1}{z+1}}$$

### 證明

在單位圓上 $z=e^{j\omega T}$，代入並分子分母同除 $e^{j\omega T/2}$：

$$w=\frac{2}{T}\cdot\frac{e^{j\omega T/2}-e^{-j\omega T/2}}{e^{j\omega T/2}+e^{-j\omega T/2}}=j\frac{2}{T}\tan\frac{\omega T}{2}$$

**是純虛數 → 單位圓確實映射成虛軸。** 而且定義了 **$w$ 平面頻率**：

$$\boxed{\omega_w=\frac{2}{T}\tan\frac{\omega T}{2}}$$

**低頻時**（$\tan x\approx x$）$\omega_w\approx\omega$，誤差 4% 以內的條件是：

$$\omega\le\frac{\omega_s}{10}$$

> **💡 教材的實務建議**：把 $T$ 選成讓這個條件在**整個系統頻寬內**都成立。因為在 $\omega=\omega_s/10$ 時，**ZOH 已經引入 $18^\circ$ 的相位落後**（第 3 章 Fig. 3-13），這個量足以大幅影響穩定度。

### 程式怎麼做

$$\texttt{Gw = d2c(Gz, 'tustin')}$$

**`'tustin'` 就是雙線性轉換的另一個名字。** `d2c` 回傳的是 $s$ 的函數，但在本章的記號中它代表 $w$。


In [ ]:
%% 實驗 1：雙線性轉換與 Routh-Hurwitz（例 7.2，T = 0.1 s）
T1  = 0.1;
Gz1 = tf([0.00484 0.00468], [1 -1.905 0.905], T1);
Gw1 = d2c(Gz1, 'tustin');          % 'tustin' = 雙線性轉換
[nw1, dw1] = tfdata(Gw1, 'v');

printf('G(w) 分子 = [%.4e  %.5f  %.4f]\n', nw1);
printf('G(w) 分母 = [%g  %.4f  %.3e]\n', dw1);
printf('教材 (-4.199e-05 w^2 - 0.04913 w + 0.9995)/(w^2 + 0.9974 w + ~0)\n\n');

% 特徵方程式 1+K*G(w)=0 -> (1-0.000042K)w^2 + (0.997-0.0491K)w + K = 0
% Routh 陣列（二階很簡單：所有係數同號即穩定）
printf('Routh 陣列（1 + K*G(w) = 0）：\n');
printf('  w^2 | 1 - 0.000042K    K        ->  K < %.0f\n', 1/0.000042);
printf('  w^1 | 0.997 - 0.0491K           ->  K < %.1f   <- 最嚴格\n', 0.997/0.0491);
printf('  w^0 | K                         ->  K > 0\n');
printf('=> 穩定範圍 0 < K < %.1f   教材 20.3\n', 0.997/0.0491);

### 結果解讀

`d2c(Gz,'tustin')` 算出的 $G(w)$ 與教材**每一位數字都吻合**。

**Routh 陣列對二階系統特別簡單**——只要**第一行的三個係數全部同號**（都為正）就穩定。三個條件中 $w^1$ 列最嚴格，給出 $K<20.3$。

---


## 四、取樣週期對穩定度的影響 (例 7.3)

**同一個受控體 $G_p(s)=\dfrac{1}{s(s+1)}$，只是把取樣週期從 $0.1$ 秒改成 $1$ 秒。**

$$G(w)=\frac{-0.03801w^2-0.386w+0.924}{w^2+0.924w}$$

$$1+KG(w)=(1-0.03801K)w^2+(0.924-0.386K)w+0.924K=0$$

Routh 陣列的 $w^1$ 列給出 $K<2.39$。


In [ ]:
%% 實驗 2：取樣週期的影響（例 7.3，T = 1 s）
T2  = 1;
Gz2 = tf([0.368 0.264], [1 -1.368 0.368], T2);
Gw2 = d2c(Gz2, 'tustin');
[nw2, dw2] = tfdata(Gw2, 'v');

printf('G(w) 分子 = [%.5f  %.4f  %.4f]   教材 [-0.03801 -0.386 0.924]\n', nw2);
printf('G(w) 分母 = [%g  %.4f  %.3e]     教材 [1 0.924 ~0]\n\n', dw2);

printf('【本節最重要的結論】\n');
printf('  T = %.1f s  ->  0 < K < %.1f\n', T1, 0.997/0.0491);
printf('  T = %.1f s  ->  0 < K < %.2f\n', T2, 0.924/0.386);
printf('  T 變大 10 倍，可用增益縮小約 %.1f 倍\n', (0.997/0.0491)/(0.924/0.386));

### 結果解讀

$$\boxed{T\text{ 從 }0.1\text{ 增到 }1\text{ 秒，可用增益從 }20.3\text{ 掉到 }2.39\text{（縮小 8.5 倍）}}$$

> **穩定度隨 $T$ 增大而劣化，原因是取樣器與資料保持器引入的相位落後。**
>
> 這與第 3 章實驗 9（ZOH 等效 $T/2$ 延遲）與第 6 章例 6.6（阻尼比從 0.5 掉到 0.25）**講的是同一件事，只是從三個不同角度看**：
>
> | 章節 | 觀察到的現象 |
> |---|---|
> | 第 3 章 | ZOH 的線性相位 $=$ 純延遲 $T/2$ |
> | 第 6 章 | $T$ 變大 → 阻尼比 $\zeta$ 下降 |
> | **第 7 章** | **$T$ 變大 → 可用增益範圍縮小** |

---


## 五、Jury 穩定度檢定 (7.5)

### 為什麼有這個方法

> **Routh–Hurwitz 要先做雙線性轉換才能用。Jury 檢定「直接」用在寫成 $z$ 的特徵方程式上。**

### 條件（式 7-15）

$Q(z)=a_nz^n+\cdots+a_0=0$（$a_n>0$）**沒有根在單位圓上或圓外**的充要條件：

$$Q(1)>0,\qquad(-1)^nQ(-1)>0,\qquad|a_0|<a_n,\qquad|b_0|>|b_{n-1}|,\ \dots$$

> **陣列大小**：**二階系統的陣列只有一列**（所以只要前三個條件）。每增加一階多兩列，$n$ 階共 $n+1$ 個條件。

### 教材建議的使用步驟

| 步驟 | 內容 |
|---|---|
| 1 | 先檢查前三個條件——**它們不需要任何計算**。任一個不滿足就停止 |
| 2 | 建構陣列，每算一列就檢查對應條件 |

### 例 7.4

$$z^2+(0.368K-1.368)z+(0.368+0.264K)=0$$

**二階，只有三個條件，都不必建陣列**：

| 條件 | 展開 | 結果 |
|---|---|---|
| $Q(1)>0$ | $0.632K>0$ | $K>0$ |
| $(-1)^2Q(-1)>0$ | $2.736-0.104K>0$ | $K<26.3$ |
| $| a_0|<a_2$ | $0.368+0.264K<1$ | $K<2.39$ ← **最嚴格** |


In [ ]:
%% 實驗 3：Jury 穩定度檢定（例 7.4）
printf('特徵方程式：z^2 + (0.368K-1.368)z + (0.368+0.264K) = 0\n');
printf('二階系統只有 n+1 = 3 個條件，都不必建陣列：\n');
printf('  Q(1) > 0         : 0.632K > 0          ->  K > 0\n');
printf('  (-1)^2 Q(-1) > 0 : 2.736 - 0.104K > 0  ->  K < %.1f\n', 2.736/0.104);
printf('  |a0| < a2        : 0.368 + 0.264K < 1  ->  K < %.2f  <- 最嚴格\n\n', 0.632/0.264);
Kc = 0.632/0.264;

% 數值驗證：直接求根，看最大絕對值有沒有超過 1
printf('數值驗證（直接求特徵方程式的根）：\n');
printf('  K        最大 |極點|    穩定?\n');
for Kx = [1 2 2.39 2.4 3]
    p = roots([1, 0.368*Kx-1.368, 0.368+0.264*Kx]);
    printf('  %-8.2f %-14.6f %s\n', Kx, max(abs(p)), merge(max(abs(p)) < 1, '是', '否'));
end
printf('\n=> Jury 給的 K < %.2f 與直接求根的結果完全一致\n', Kc);

### 結果解讀

Jury 給的 $K<2.39$ 與直接求根的結果完全吻合（$K=2.39$ 時最大 $|$極點$|=0.9995$，$K=2.4$ 時變成 $1.0008$）。

### 💡 Jury vs Routh–Hurwitz 怎麼選

| | Routh–Hurwitz | Jury |
|---|---|---|
| 用在哪 | **$w$ 平面**（要先轉換） | **$z$ 平面**（直接用） |
| 二階系統 | 要轉換 + 建陣列 | **三個條件，不必建陣列** |
| 高階系統 | 陣列較短 | 陣列較長（每階兩列） |

**低階系統用 Jury 比較快。但要做 Bode 或 Nyquist 分析時，無論如何都要轉到 $w$ 平面。**

---


## 六、由臨界增益求振盪頻率

**臨界穩定時系統會等幅振盪，振盪頻率可以用兩種方法求。**

### 方法一：$z$ 平面（配合 Jury）

$K=2.39$ 時 $z^2-0.488z+1=0$，根為

$$z=0.244\pm j0.970=1\angle(\pm75.9^\circ)=1\angle(\pm1.32\text{ rad})=1\angle(\pm\omega T)$$

**因為 $| z|=1$（在單位圓上）**，而 $z=e^{j\omega T}$，所以 $\omega=\theta/T=1.32$ rad/s。

### 方法二：$w$ 平面（配合 Routh–Hurwitz）

由 $w^2$ 列寫出**輔助方程式**：

$$(1-0.03801K)w^2+0.924K\Big|_{K=2.39}=0.9092w^2+2.2084=0\ \Longrightarrow\ \omega_w=1.5585$$

再由式 (7-10) 反解：

$$\omega=\frac{2}{T}\tan^{-1}\frac{\omega_wT}{2}=1.324\ \text{rad/s}$$

**兩種方法應該給出相同答案。**


In [ ]:
%% 實驗 4：振盪頻率（例 7.3、7.4）
% 方法一：z 平面
p = roots([1, 0.368*Kc-1.368, 0.368+0.264*Kc]);
printf('[z 平面] K = %.2f 時的根：\n', Kc);
printf('  z = %.3f +- j%.3f，|z| = %.4f（應為 1）\n', real(p(1)), abs(imag(p(1))), abs(p(1)));
printf('  角度 = %.1f deg = %.4f rad   教材 75.9 deg / 1.32 rad\n', ...
       abs(angle(p(1)))*180/pi, abs(angle(p(1))));
printf('  振盪頻率 w = theta/T = %.4f rad/s   教材 1.32\n\n', abs(angle(p(1)))/T2);

% 方法二：w 平面輔助方程式
ww = sqrt(0.924*Kc/(1 - 0.03801*Kc));
printf('[w 平面] 輔助方程式 (1-0.03801K)w^2 + 0.924K = 0\n');
printf('  w_w = %.4f   教材 1.5585\n', ww);
printf('  由式(7-10)反解 w = (2/T)atan(w_w*T/2) = %.4f rad/s   教材 1.324\n', ...
       (2/T2)*atan(ww*T2/2));
printf('\n=> 兩種方法一致\n');

### 結果解讀

兩種方法都給出 $\omega=1.3248$ rad/s，與教材的 $1.32$／$1.324$ 吻合。

> **💡 為什麼 $| z|$ 恰好是 1**：臨界穩定的定義就是「根落在單位圓上」。而單位圓上的點 $z=e^{j\omega T}$，其輻角除以 $T$ 就是振盪頻率——這正是第 6 章式 (6-7) 的內容。

---


## 七、根軌跡 (7.6)

### 教材的關鍵論點

> **離散系統的根軌跡建構規則與連續系統「完全相同」**——因為任何方程式的根**只取決於係數**，與變數叫什麼名字無關。

### ⚠️ 但解讀完全不同

| | $s$ 平面 | $z$ 平面 |
|---|---|---|
| 穩定區 | 左半平面 | **單位圓內** |
| 極點位置的時間響應意義 | 見連續系統教科書 | **見第 6 章 Fig. 6-11** |

### 分離點怎麼算（規則 6）

$$\frac{d[\overline{GH}(z)]}{dz}=0\quad\Longleftrightarrow\quad
\text{den}\cdot\frac{d\,\text{num}}{dz}-\text{num}\cdot\frac{d\,\text{den}}{dz}=0$$

**程式寫法**：用 `polyder`（多項式微分）與 `conv`（多項式相乘）：

```matlab
brk = roots(conv(den, polyder(num)) - conv(num, polyder(den)));
```

> **比數值掃描好在哪**：這是**解析解**，直接給出精確的根；數值掃描 $\frac{d}{dz}$ 的變號點會在**極點附近產生假的分離點**（因為那裡函數本身是奇異的）。

### 例 7.7 的預期結果

| 項目 | 值 |
|---|---|
| 軌跡起點 | $z=1,\ 0.368$（極點） |
| 軌跡終點 | $z=-0.717$（零點）與 $\infty$ |
| 分離點 | $z=0.65$（$K=0.196$）與 $z=-2.08$（$K=15.0$） |
| 穿越單位圓 | $z=0.244\pm j0.970$，$K=2.39$ |


In [ ]:
%% 實驗 5：根軌跡（例 7.7）
[nz2, dz2] = tfdata(Gz2, 'v');
printf('KG(z) = 0.368K(z + 0.717)/((z-1)(z-0.368))\n');
printf('  零點 z = %.4f          教材 -0.717\n', roots(nz2));
printf('  極點 z = %.4f, %.4f  教材 1 與 0.368\n\n', roots(dz2));

% 分離點：解析解（規則 6 的等價形式）
nb = [0.368 0.264]; db = [1 -1.368 0.368];
brk = roots(conv(db, polyder(nb)) - conv(nb, polyder(db)));
brk = sort(real(brk(abs(imag(brk)) < 1e-9)));
printf('分離點（解 den*num'' - num*den'' = 0）：\n');
for b = brk'
    Kb = -1/(polyval(nb,b)/polyval(db,b));
    printf('  z = %8.4f  ->  K = %.4f\n', b, Kb);
end
printf('教材：z=0.65 時 K=0.196；z=-2.08 時 K=15.0\n');

figure('Position', [50 50 620 560]);
rlocus(Gz2); hold on;
th = linspace(0, 2*pi, 400);
plot(cos(th), sin(th), 'k--', 'LineWidth', 1.2);      % 單位圓 = 穩定邊界
plot(real(p), imag(p), 'rp', 'MarkerSize', 14, 'MarkerFaceColor', 'r');
plot(brk, zeros(size(brk)), 'gs', 'MarkerSize', 10, 'MarkerFaceColor', 'g');
axis([-3 2 -2 2]); grid on;
title('根軌跡（例 7.7）：紅星 = K=2.39 穿越點，綠方 = 分離點');
xlabel('Re(z)'); ylabel('Im(z)');

### 結果解讀

分離點的解析解 $z=-2.0827$（$K=15.04$）與 $z=0.6479$（$K=0.196$）**與教材完全吻合**。

**圖上三種標記的意義**：

| 標記 | 意義 |
|---|---|
| 黑色虛線圓 | **單位圓 = 穩定邊界**（$s$ 平面對應虛軸） |
| 綠色方塊 | **分離點**——兩條實軸軌跡在此離開實軸變成共軛複數 |
| 紅色星號 | **$K=2.39$ 穿越單位圓處**，系統從穩定變不穩定 |

> **💡 怎麼從根軌跡讀設計資訊**：軌跡從極點（$K=0$）出發往零點跑。$K$ 越大極點跑得越遠——本例往單位圓外跑，所以 $K$ 有上限。**第 8 章的設計就是在調整零極點位置，把軌跡「拉」到你要的地方。**

---


## 八、Nyquist 準則 (7.7)

### 判準

$$\boxed{Z=N+P}$$

| 符號 | 意義 |
|---|---|
| $N$ | Nyquist 圖**順時針包圍 $-1$ 點的次數** |
| $P$ | 開迴路在 Nyquist 路徑**外**的極點數 |
| $Z$ | **閉迴路**在單位圓外的極點數 |

**系統穩定 $\iff Z=0$。**

### 三種等價算法（Table 7-4）

$\overline{GH}^*(s)$、$\overline{GH}(z)$、$\overline{GH}(w)$ **三者畫出的 Nyquist 圖完全相同**。

### ⚠️ 兩個實軸交點的位置

教材說例 7.10 的交點是 $-0.0381$ 與 $-0.418$。但實作時要注意：

| 交點 | 在哪裡 | 怎麼找 |
|---|---|---|
| $-0.418$ | 曲線中間**穿越**實軸 | 掃頻找 $\mathrm{Im}=0$ 的變號點 |
| $-0.0381$ | **曲線端點**（$\omega=\pi/T$，即 $z=-1$） | **掃頻找不到！要直接代 $z=-1$** |

> **為什麼 $\omega=\pi/T$ 時 $z=-1$**：$z=e^{j\omega T}$，代入 $\omega=\pi/T$ 得 $z=e^{j\pi}=-1$。這是 **Nyquist 頻率**（$\omega_s/2$），也是離散系統頻率響應的**上限**——超過它就是混疊（第 3 章）。


In [ ]:
%% 實驗 6：Nyquist 圖（例 7.10）
wv = logspace(-3, log10(pi/T2), 40000);
Lv = polyval(nz2, exp(1j*wv*T2)) ./ polyval(dz2, exp(1j*wv*T2));
re = real(Lv); im = imag(Lv);

% 交點一：曲線中間穿越實軸
idx = find(im(1:end-1).*im(2:end) < 0);
% 交點二：端點 w = pi/T，也就是 z = -1（掃頻找不到，要直接代）
G_at_minus1 = polyval(nz2, -1) / polyval(dz2, -1);

printf('穿越實軸的交點：'); printf('%.4f  ', re(idx)); printf('\n');
printf('端點 w = pi/T（z = -1）：G(-1) = %.4f\n', G_at_minus1);
printf('教材 -0.0381 與 -0.418  => 兩者都對上了\n\n');

xc = min([re(idx), G_at_minus1]);
printf('由最負交點 %.4f 推得臨界增益 = 1/|%.4f| = %.3f   教材 2.39\n', xc, xc, 1/abs(xc));
printf('判別：N = 0（不包圍 -1 點），P = 0  =>  Z = N+P = 0，系統穩定\n');

figure('Position', [50 50 700 420]);
plot(re, im, 'b-', 'LineWidth', 1.5); hold on;
plot(re, -im, 'b-', 'LineWidth', 1.5);
plot(-1, 0, 'r+', 'MarkerSize', 16, 'LineWidth', 2);
plot([xc G_at_minus1], [0 0], 'ko', 'MarkerFaceColor', 'k', 'MarkerSize', 6);
grid on; axis([-1.2 0.2 -0.7 0.7]);
title('Nyquist 圖（例 7.10）：紅十字 = -1 點，黑點 = 實軸交點');
xlabel('Re'); ylabel('Im');

### 結果解讀

兩個交點都對上了教材的 $-0.0381$ 與 $-0.418$。

**臨界增益 $=\dfrac{1}{0.418}=2.39$**——與 Routh–Hurwitz、Jury、根軌跡**三種方法的結果完全一致**。這就是教材說的「用同一個系統貫穿全章，提供各種技巧之間的共同比較基準」。

> **💡 教材的重要對照**：**同一個系統若「沒有取樣」，對所有正增益都穩定**。取樣的去穩定化效果來自取樣器與保持器的相位落後。
>
> **⚠️ 但教材也警告**：類比模型對所有正增益都穩定，**這不能推廣到實體系統**——二階模型只在有限訊號準位內準確，**大增益會產生大訊號，線性模型就不再成立了**。

---


## 九、Bode 圖與裕度 (7.8)

### ⚠️ 為什麼 Bode 圖「必須」用 $w$ 平面

> **Bode 圖的直線近似，基礎是「自變數 $j\omega$ 是純虛數」。因此離散系統的 Bode 圖若要用直線近似，「必須」使用 $w$ 平面形式。**

### 例 7.12

$$G(w)=\frac{-0.0381(w-2)(w+12.14)}{w(w+0.924)}$$

**轉折頻率**：$0$（積分器）、$0.924$（分母）、$2$ 與 $12.14$（分子）。

### 增益裕度與相位裕度

| 名詞 | 定義 | 意義 |
|---|---|---|
| **增益裕度** | 相位 $=-180^\circ$ 處，增益還差多少到 0 dB | **還可以放大幾倍才不穩定** |
| **相位裕度** | 增益 $=0$ dB 處，相位還差多少到 $-180^\circ$ | **還可以多幾度相位落後才不穩定** |

> **在 Bode 圖上，增加增益 = 把整條大小曲線垂直往上平移。若增加的量等於增益裕度，系統就臨界穩定。**

### ⚠️ `margin` 的回傳值是「倍率」不是 dB

```matlab
[gm, pm, wg, wp] = margin(sys);
gm_dB = 20*log10(gm);     % 要自己換算
```


In [ ]:
%% 實驗 7：Bode 圖與裕度（例 7.12）
Gw_ex = tf([-0.0381 -0.386 0.924], [1 0.924 0]);
printf('G(w) 零點 = %.3f, %.3f   教材 (w-2)(w+12.14)\n', sort(roots([-0.0381 -0.386 0.924])));
printf('G(w) 極點 = %.3f, %.3f   教材 w(w+0.924)\n', sort(roots([1 0.924 0])));
printf('轉折頻率：0, 0.924, 2, 12.14\n\n');

[gm_z, pm_z] = margin(Gz2);
[gm_w, pm_w] = margin(Gw_ex);
printf('由 G(z) 算：GM = %.4f 倍 = %.2f dB，PM = %.2f deg\n', gm_z, 20*log10(gm_z), pm_z);
printf('由 G(w) 算：GM = %.4f 倍 = %.2f dB，PM = %.2f deg\n', gm_w, 20*log10(gm_w), pm_w);
printf('=> 兩者相同（同一個系統的兩種表示）\n');
printf('=> 增益裕度 %.3f 倍，正好就是前面四種方法都算出的臨界增益 K = 2.39\n', gm_z);

figure('Position', [50 50 780 520]);
bode(Gw_ex); grid on;
title('Bode 圖：G(w)（例 7.12）');

### 結果解讀

$$\boxed{\text{增益裕度}=2.394\text{ 倍}=\text{臨界增益 }K=2.39}$$

**這不是巧合**——增益裕度的定義就是「還可以放大幾倍才不穩定」，而臨界增益就是「放大到多少會不穩定」。**兩者本來就是同一個數字。**

### 五種方法的總結

本章用**同一個系統**（$G_p(s)=\frac{1}{s(s+1)}$，$T=1$ s）示範了五種方法，**全部得到 $K_{\text{crit}}=2.39$**：

| 方法 | 在哪個平面 | 怎麼得到 2.39 |
|---|---|---|
| Routh–Hurwitz | $w$ | $w^1$ 列：$0.924-0.386K>0$ |
| Jury | $z$ | $| a_0|<a_2$：$0.368+0.264K<1$ |
| 根軌跡 | $z$ | 軌跡穿越單位圓處的增益 |
| Nyquist | $z$ | $1/0.418$ |
| **Bode** | $w$ | **增益裕度 $=2.394$** |

---


## 十、⚠️ 三個實測發現的 Octave 陷阱

**這一節是本 Notebook 的補充內容**（教材沒有）。這三個陷阱都是實測發現的，會直接影響第 8 章的設計計算。


In [ ]:
%% 附錄：三個 Octave 陷阱
% ---- 陷阱 1：bode 對含 z=1 極點的離散系統，相位錯誤 ----
Tp = 0.05;
Gp_ex8 = c2d(tf([2],[1 3 2 0]), Tp, 'zoh');    % 第 8 章的受控體，含積分器
printf('陷阱1：bode 對「含 z=1 極點」的離散系統，相位是錯的\n');
printf('  w       bode 相位    freqresp 相位   教材 Table 8-1\n');
book = [-98.7 -120.5 -163.0]; i = 1;
for w = [0.1 0.36 1.0]
    [~, pb] = bode(Gp_ex8, w);
    H = freqresp(Gp_ex8, w);
    printf('  %-7.2f %-12.2f %-15.2f %.1f\n', w, pb, angle(H)*180/pi, book(i));
    i = i + 1;
end
printf('\n  對照：沒有 z=1 極點的系統則完全正確\n');
Gok = tf(1,[1 -0.99],1);
[~, pb] = bode(Gok, 0.5); H = freqresp(Gok, 0.5);
printf('    1/(z-0.99) 在 w=0.5：bode %.2f，freqresp %.2f，差 %.1f\n', ...
       pb, angle(H)*180/pi, pb-angle(H)*180/pi);
printf('  => 取相位一律用 freqresp\n\n');

% ---- 陷阱 2：margin 有時找不到增益交越頻率 ----
printf('陷阱2：margin 有時找不到增益交越頻率\n');
Dz_lag = 0.3890*tf([1 -0.9982],[1 -0.9993], Tp);    % 第 8 章例 8.1 的補償器
L = Dz_lag*Gp_ex8;
[gm, pm, wg, wp] = margin(L);
printf('  margin(L) -> GM = %.2f dB，PM = %.2f deg @ %.4f rad/s\n', 20*log10(gm), pm, wp);
% 自己掃頻算
[nL, dL] = tfdata(L, 'v');
wv2 = logspace(-4, log10(pi/Tp), 200000);
Lv2 = polyval(nL, exp(1j*wv2*Tp))./polyval(dL, exp(1j*wv2*Tp));
mag = abs(Lv2); ph = unwrap(angle(Lv2))*180/pi;
i1 = find(mag(1:end-1)>=1 & mag(2:end)<1, 1);
printf('  手算：|L|=1 @ w=%.4f，相位 %.2f  ->  PM = %.2f deg （教材 55.9）\n', ...
       wv2(i1), ph(i1), 180+ph(i1));
printf('  => margin 回傳 pm=180 或 wp=NaN 時，自己掃頻算\n\n');

% ---- 陷阱 3：exist 判斷不了 LTI 方法 ----
printf('陷阱3：exist 判斷不了 LTI 類別的「方法」\n');
for f = {'d2c', 'c2d', 'margin', 'nyquist'}
    printf('  exist(''%s'') = %d，但實際可以呼叫\n', f{1}, exist(f{1}));
end
printf('  => 不要用 exist 判斷 c2d/d2c/margin/nyquist 是否可用\n');

### 陷阱的影響有多大

**陷阱 1 最嚴重**——它會讓**第 8 章的設計程序算出完全錯誤的答案**。

第 8 章例 8.2 的設計步驟是：

$$\phi=180^\circ+\eta_m-\angle G(j\omega_{w1})$$

| 用什麼取相位 | $\angle G$ | $\phi$ | 約束 3（$\cos\phi>a_0| G|$） |
|---|---|---|---|
| **`freqresp`（正確）** | $-172.88^\circ$ | $47.9^\circ$ | $0.671>0.457$ ✓ **通過** |
| `bode`（錯誤） | $+7.12^\circ$ | $227.9^\circ$ | $-0.671>0.457$ ✗ **失敗** |

**用 `bode` 取相位，設計會直接失敗。** 這就是為什麼第 8 章的所有程式都改用 `freqresp`。

> **注意**：教材的 MATLAB 程式用的是 `[mag,phase]=bode(Gz,ww1)`。**在 MATLAB 上可能沒問題，但在 Octave 上必須改用 `freqresp`。**

---


## 十一、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (7-3) | $1+\overline{GH}(z)=0$ | **特徵方程式** | 3 |
| — | $1-G_{op}(z)=0$ | 寫不出轉移函數時 | — |
| (7-7)(7-8) | $z=\frac{1+(T/2)w}{1-(T/2)w}$ | **雙線性轉換** | 1 |
| (7-10) | $\omega_w=\frac{2}{T}\tan\frac{\omega T}{2}$ | $w$ 平面頻率 ↔ 真實頻率 | 4 |
| (7-12) | $\omega\le\omega_s/10$ | $\omega_w\approx\omega$ 的有效範圍 | — |
| Table 7-1 | Routh 陣列 | 在 $w$ 平面判穩定 | 1, 2 |
| (7-15) | Jury 條件 | 直接在 $z$ 平面判穩定 | 3 |
| Table 7-3 | 根軌跡規則 | 與連續系統相同 | 5 |
| — | $Z=N+P$ | Nyquist 準則 | 6 |

### MATLAB / Octave 常見錯誤

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| **用 `bode` 取離散系統的相位** | 含 $z=1$ 極點時錯 $90\sim180^\circ$ | **`freqresp`** |
| 直接信任 `margin` 的 PM | 有時回傳 180 @ NaN | 自己掃頻驗算 |
| 用 `exist` 判斷 `c2d`/`d2c` | 回傳 0 但函數可用 | 直接 try 呼叫 |
| 把 `margin` 的 `gm` 當 dB | 差很多 | `20*log10(gm)` |
| 數值掃描找分離點 | 極點附近出現假分離點 | 用 `polyder` 解析解 |
| 掃頻找 Nyquist 實軸交點 | 漏掉 $\omega=\pi/T$ 的端點 | 額外代 $z=-1$ |
| 忘記 `unwrap` 相位 | 相位在 $\pm180$ 處跳變，裕度算錯 | `unwrap(angle(L))` |

### 承先啟後

本章建立了**判斷穩定度的五種工具**，而且用同一個系統證明它們給出相同答案（$K_{\text{crit}}=2.39$）。

**第 8 章把這些分析工具轉成設計工具**：

| 第 7 章（分析） | 第 8 章（設計） |
|---|---|
| 算出目前的相位裕度 | **設計 $D(z)$ 讓相位裕度達到 $55^\circ$** |
| 看根軌跡怎麼跑 | **加零極點把軌跡「拉」到想要的地方** |
| 讀 Bode 圖的轉折頻率 | **放置補償器的轉折頻率** |

**本章的雙線性轉換是第 8 章的基礎**——所有頻率響應設計都在 $w$ 平面進行，設計完再用式 (8-14) 轉回 $z$ 平面。
